    Получаем массива торговых инструментов с MetaTrader 5 server
        Изменение значений полей ОПИСАНИЕ и МЕЖДУНАРОДНОЕ ОПИСАНИЕ на MetaTrader 5 server
        Изменение значений хеджированой маржи
##### <span style="color:red">Тесты отсутствуют !</span>

In [1]:
# Инициализация необходимых функций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
import os, sys
import pandas as pd
import numpy as np
current_dir = os.getcwd()                                               # Определяем путь к текущему файлу и его родительской директории
parent_dir = os.path.dirname(current_dir)                               # Формируем путь к libraries_py
libraries_path = os.path.join(parent_dir, "libraries_py")               # Формируем путь к libraries_py
sys.path.append(libraries_path)                                         # Добавляем libraries_py в sys.path

print (f"Обращаемся к файлк [config_loader.py], в директории {libraries_path}"),
from config_loader import load_config, setup_libraries, import_dynamic_functions

directories = load_config()
libraries_path = setup_libraries(directories)
import_functions, print_import_function_info = import_dynamic_functions(libraries_path)

if import_functions and print_import_function_info:
    modules_to_import = {
                        'mt5_api': [libraries_path,
                            'manager_connect_with_control',
                            'admin_connect_with_control',
                            'admin_disconnect_with_control',
                            'getting_array_trading_instruments'
                                 ],
                        "sed_array_lib": [libraries_path,
                                          "np_set_printoptions",
                            "array_to_dataframe_with_multiline_headers",
                                      "array_to_dataframe"
                                      ]}                    # ключи — названия модулей, значения — списки функций
    
    imported = import_functions(modules_to_import)
    print_import_function_info(modules_to_import, imported)

    # Получаем нужные переменные
    directory_data_temp_files = directories.get("directory_data_temp_files", None)
    directory_data_log_files = directories.get("directory_data_log_files", None)
    if not directory_data_temp_files:
        raise ValueError("❌ ERROR: directory_data_temp_files не найден в конфигурации!")
    
else: print("❌ Ошибка при импорте функций.")

#imported_functions = dynamic_import_from_modules(modules_to_import)             # Импортируем функции по словарю

Обращаемся к файлк [config_loader.py], в директории c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\libraries_py
Рабочая директория проекта c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations
📂 directory_data_temp_files: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\working_data_files
📂 directory_data_log_files: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\log_data_files
📂 directory_libraries_path: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\libraries_py
✅ Каталог c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\libraries_py успешно добавлен в sys.path

 ✅ Импорт [dynamic_import_functions.py] успешен.
Импорт из 'mt5_api' успешен: ['manager_connect_with_control', 'admin_connect_with_control', 'admin_disconnect_with_control', 'getting_array_trading_instruments']
Импорт из 'sed_array_lib' успешен: ['np_set_printoptions', 'array_to_dataframe_with_multiline_headers', 'array_to_dataframe']

 Импортированные фун

In [3]:
# Получаем массив торговых инструментов
symbol_array = imported["getting_array_trading_instruments"]('*')

if symbol_array:
    print(dir(symbol_array[0]))  # Покажет доступные атрибуты объекта

<getting_array_trading_instruments>: symbols_list =  *
MT5manager connect: True
SymbolTotal = 0
len_symbol_array =  4463
manager.Disconnect() =  True
['AccruedInterest', 'Basis', 'CFI', 'CalcMode', 'Category', 'ChartMode', 'Clear', 'Color', 'ColorBackground', 'ContractSize', 'Country', 'CurrencyBase', 'CurrencyBaseDigits', 'CurrencyMargin', 'CurrencyMarginDigits', 'CurrencyProfit', 'CurrencyProfitDigits', 'Description', 'Digits', 'EnCalcMode', 'EnChartMode', 'EnExecutionMode', 'EnExpirationFlags', 'EnFillingFlags', 'EnGTCMode', 'EnIndustries', 'EnInstantFlags', 'EnInstantMode', 'EnMarginFlags', 'EnMarginRateTypes', 'EnOptionMode', 'EnOrderFlags', 'EnRequestFlags', 'EnSectors', 'EnSpliceTimeType', 'EnSpliceType', 'EnSwapDays', 'EnSwapFlags', 'EnSwapMode', 'EnTickFlags', 'EnTradeFlags', 'EnTradeMode', 'Exchange', 'ExecMode', 'ExpirFlags', 'FaceValue', 'FillFlags', 'FilterDiscard', 'FilterGap', 'FilterGapTicks', 'FilterHard', 'FilterHardTicks', 'FilterSoft', 'FilterSoftTicks', 'FilterSpre

In [ ]:
# Изменяем значение хеджированной маржи <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
for a in symbol_array:
    try:
        contract_size = a.ContractSize
        margin_hedged = int(contract_size * 2)
        print(f"✅ Symbol: [ {a.Description} ], Contract Size: {contract_size}, Margin Hedged: {margin_hedged}")
        a.MarginHedged = margin_hedged
    except Exception as e:
        print(f"❌ ERROR: Не удалось изменить [MarginHedged] для символа {getattr(a, 'Symbol', 'Unknown')} — {e}")

ПРИМЕР ВЫВОДА:

    Изменнение значений полей ОПИСАНИЕ и МЕЖДУНАРОДНОЕ ОПИСАНИЕ <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

In [ ]:
# Замена Международного названия символа на Описание
for a in symbol_array:
    if a.Path.startswith('CRYPTO'): a.International = a.Description

In [ ]:
# Замена описания символа на имя символов
for a in symbol_array:
    if a.Path.startswith('CRYPTO\\Crypto'): a.Description = a.Symbol

In [ ]:
# Выделяем Символвы, Определяем концовку которую нужно обрезать # Обрезаем последние два символа в ОПИСАНИИ
for a in symbol_array:                                                      # Проход по элементам массива
    if a.Path.startswith('FOREX'):
        if a.Description.endswith('.i'): a.Description = a.Description[:-2]

In [ ]:
# Вставляем " / " посередине 
for a in symbol_array:
    if a.Path.startswith('CRYPTO\\Crypto - USDT'):
        if len(a.Description) > 1:
            a.Description = a.Description[:-4] + ' / ' + a.Description[-4:]

In [ ]:
# удаление символа "/", если он присутствует в строке
for a in symbol_array:
    if a.Path.startswith('CRYPTO\\Crypto - USDT'):
        a.Description = a.Description.replace(" / ", "")

In [ ]:
# элементы из массива symbol_array, у которых значение Description встречается более одного раза
"""Чтобы отобрать элементы из массива symbol_array, у которых значение Description встречается более одного раза,
    и составить список значений Symbol этих элементов, можно выполнить следующие шаги:
    Пройтись по массиву и собрать количество вхождений каждого значения Description.
    Отобрать те Description, которые встречаются более одного раза.
    Составить список соответствующих Symbol.
"""
description_count = {}                              # Словарь для подсчета вхождений Description
for a in symbol_array:                              # Подсчет вхождений Description
    if a.Path.startswith('FOREX'):
        description = a.Description
        if description in description_count:
            description_count[description] += 1
        else:
            description_count[description] = 1

symbols = []                                        # Список для хранения Symbol значений
for a in symbol_array:                              # Отбор Symbol для повторяющихся Description
    if a.Path.startswith('FOREX') and description_count.get(a.Description, 0) > 1:
        symbols.append(a.Symbol)

print("Колличество элементов symbol_array, у которых значение Description встречается более одного раза:", len(symbols), "\n",
      "Список:", symbols)

In [13]:
# Обновление массива символов на сервере <<<<<<<<<<<<<<<<<<<<<<<<<<<< 
admin = imported["admin_connect_with_control"](admin if 'admin' in locals() else None)
if admin:
    print("SymbolUpdateBatch = ", admin.SymbolUpdateBatch(symbol_array))
    print("manager.Disconnect() = ", admin.Disconnect())
else: print(f"MT5Admin Failed to connect to server: {MT5Manager.LastError()}")        # не удалось подключиться к серверу           

if imported["admin_disconnect_with_control"](admin): del admin
else: print("❌ ERROR: разъединение mt5admin c сервером НЕ удалось.")

admin.Disconnect() True
Присутствовало не завешенное соединение MT5 администратор
MT5admin connect: True
SymbolUpdateBatch =  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 